In [1]:

import os


def repo_root():
    cur = os.getcwd()
    for _ in range(8):
        if os.path.isdir(os.path.join(cur, 'data')) and os.path.exists(
            os.path.join(cur, 'data', 'appointments.csv')
        ):
            return cur
        nxt = os.path.dirname(cur)
        if nxt == cur:
            break
        cur = nxt
    raise FileNotFoundError('Could not locate the repository root')


ROOT = repo_root()
OUTPUT_DIR = ROOT
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Repository root:', ROOT)


Repository root: /Users/harish/careloop-clinics-no-show-guard


## Task 3.1 — Feature selection and baseline

Objective: build the modelling frame from `appointments_clean.csv`, encode features, split into train/test, and establish a baseline.

**Prediction point:** the evening before the appointment, after reminders have been sent or skipped. No future information is used, so there is no target leakage.

Features:
- `age`, `distance_km`, `fee_inr`, `lead_days`, `prior_no_shows`
- `reminder_sent`: Yes/No converted to 1/0
- `booking_channel` and `visit_type`: one-hot encoded with `pd.get_dummies`

Not used: `patient_id`, `appointment_id`, `booked_on`, `appt_date`, `clinic_id`, `gender`, `status`.

Target: `no_show`.

Split: `test_size=0.2`, `random_state=42`. The CSV row order is not changed before the split. `appointment_id` is kept separately for the final test risk-score export.

In [2]:

import pandas as pd
import numpy as np

df = pd.read_csv(os.path.join(ROOT, 'appointments_clean.csv'))
print('Records:', len(df))

# Convert reminder_sent Yes/No to 1/0.
df['reminder_sent'] = df['reminder_sent'].map({'Yes': 1, 'No': 0})

# One-hot encode booking_channel and visit_type.
X = pd.get_dummies(
    df[['age', 'distance_km', 'fee_inr', 'lead_days',
        'prior_no_shows', 'reminder_sent',
        'booking_channel', 'visit_type']],
    columns=['booking_channel', 'visit_type'],
)
y = df['no_show']

print('Feature columns:', X.shape[1])
print(X.columns.tolist())

# Keep appointment_id separately for the final test risk-score export.
appointment_ids = df['appointment_id']

# Split: 80% train, 20% test, fixed seed. Row order is not reordered beforehand.
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test, id_train, id_test = train_test_split(
    X, y, appointment_ids,
    test_size=0.2, random_state=42, shuffle=True)

print('Training records:', len(X_train))
print('Test records:', len(X_test))
print('Test no-shows:', int(y_test.sum()))
print('Test no-show rate: {:.1f}%'.format(100 * y_test.mean()))

# Baseline: predict the majority class (Attended) for every test appointment.
from sklearn.dummy import DummyClassifier
baseline = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)
print('Baseline accuracy (majority class): {:.4f}'.format(baseline.score(X_test, y_test)))


Records: 990
Feature columns: 11
['age', 'distance_km', 'fee_inr', 'lead_days', 'prior_no_shows', 'reminder_sent', 'booking_channel_App', 'booking_channel_Phone', 'booking_channel_Walk-In', 'visit_type_Consultation', 'visit_type_Follow-up']


Training records: 792
Test records: 198
Test no-shows: 49
Test no-show rate: 24.7%
Baseline accuracy (majority class): 0.7525


## Task 3.2 — Logistic regression

Objective: fit a regularised logistic regression on scaled features and evaluate it.

- `StandardScaler` fitted **only on `X_train`** (no leakage from the test set).
- `LogisticRegression(max_iter=1000)`.
- Metrics: accuracy, confusion matrix, precision, recall, F1, ROC-AUC.
- Coefficients printed beside feature names.
- `reminder_sent` odds ratios computed two ways: `np.exp(coefficient)` and `np.exp(coefficient / scaler.scale_)`, using the actual `reminder_sent` feature index.

The reminder_sent coefficient is an **association**, not evidence of causation.

In [3]:

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, precision_score,
                             recall_score, f1_score, roc_auc_score)

# Scale: fit on training data only.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_scaled, y_train)
print('Logistic regression converged:', bool(getattr(lr, 'converged_', True)))

# Predictions and probabilities on the test set.
y_pred = lr.predict(X_test_scaled)
y_proba = lr.predict_proba(X_test_scaled)[:, 1]

print()
print('=== Logistic regression — test set ===')
print('Accuracy:      {:.4f}'.format(accuracy_score(y_test, y_pred)))
print('Confusion matrix:')
print(confusion_matrix(y_test, y_pred))
print('Precision:     {:.4f}'.format(precision_score(y_test, y_pred)))
print('Recall:        {:.4f}'.format(recall_score(y_test, y_pred)))
print('F1:            {:.4f}'.format(f1_score(y_test, y_pred)))
print('ROC-AUC:       {:.4f}'.format(roc_auc_score(y_test, y_proba)))

# Coefficients beside feature names.
coef = pd.Series(lr.coef_[0], index=X_train.columns)
print()
print('Coefficients (scaled features):')
print(coef.round(4).to_string())

# reminder_sent odds ratios, two ways.
reminder_index = list(X_train.columns).index('reminder_sent')
reminder_coef = lr.coef_[0][reminder_index]
reminder_scale = scaler.scale_[reminder_index]
print()
print('reminder_sent feature index:', reminder_index)
print('reminder_sent coefficient:  {:.4f}'.format(reminder_coef))
print('np.exp(coefficient):          {:.4f}'.format(np.exp(reminder_coef)))
print('np.exp(coef / scaler.scale_): {:.4f}'.format(np.exp(reminder_coef / reminder_scale)))

print()
print('Interpretation: a scaled one-standard-deviation increase in reminder_sent is '
      'associated with lower no-show odds. Because StandardScaler re-scales each '
      'feature, the scaled coefficient is not the same as the original one-unit '
      '(No to Yes) effect; dividing the coefficient by the feature scale puts the '
      'two on a comparable basis. This is an observational association, not causal evidence.')


Logistic regression converged: True

=== Logistic regression — test set ===
Accuracy:      0.7879
Confusion matrix:
[[139  10]
 [ 32  17]]
Precision:     0.6296
Recall:        0.3469
F1:            0.4474
ROC-AUC:       0.7889

Coefficients (scaled features):
age                       -0.2539
distance_km                0.2534
fee_inr                    0.4528
lead_days                  0.6145
prior_no_shows             0.6496
reminder_sent             -0.5807
booking_channel_App       -0.0468
booking_channel_Phone      0.2159
booking_channel_Walk-In   -0.1864
visit_type_Consultation    0.0157
visit_type_Follow-up      -0.0157

reminder_sent feature index: 5
reminder_sent coefficient:  -0.5807
np.exp(coefficient):          0.5595
np.exp(coef / scaler.scale_): 0.3094

Interpretation: a scaled one-standard-deviation increase in reminder_sent is associated with lower no-show odds. Because StandardScaler re-scales each feature, the scaled coefficient is not the same as the original one-unit

/Users/harish/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/harish/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/harish/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/harish/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/harish/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: overflow encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_

## Task 3.3 — Decision trees

Objective: fit an unconstrained decision tree and a depth-limited tree, and compare training versus test accuracy to illustrate overfitting.

In [4]:

from sklearn.tree import DecisionTreeClassifier

# Unconstrained tree (prone to memorising the training set).
dt_full = DecisionTreeClassifier(random_state=42)
dt_full.fit(X_train, y_train)

# Depth-limited tree (simpler, less prone to overfitting).
dt_depth3 = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_depth3.fit(X_train, y_train)

print('=== Decision trees ===')
print('Full tree:     train accuracy = {:.4f}, test accuracy = {:.4f}'.format(
    dt_full.score(X_train, y_train), dt_full.score(X_test, y_test)))
print('Depth-3 tree:  train accuracy = {:.4f}, test accuracy = {:.4f}'.format(
    dt_depth3.score(X_train, y_train), dt_depth3.score(X_test, y_test)))

print()
print('The full tree reaches near-perfect training accuracy but scores much lower '
      'on the test set — a large training/test gap is the classic sign of '
      'overfitting (the tree memorised training noise rather than learning a '
      'generalisable rule). Constraining max_depth shrinks that gap.')


=== Decision trees ===
Full tree:     train accuracy = 1.0000, test accuracy = 0.6717
Depth-3 tree:  train accuracy = 0.8220, test accuracy = 0.7778

The full tree reaches near-perfect training accuracy but scores much lower on the test set — a large training/test gap is the classic sign of overfitting (the tree memorised training noise rather than learning a generalisable rule). Constraining max_depth shrinks that gap.


## Task 3.4 — Random forest

Objective: fit a random forest and compare it against logistic regression and the depth-3 decision tree.

Model choice must not be based on test performance alone; see Task 3.5 for cross-validation-based selection.

In [5]:

from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)
rf_proba = rf.predict_proba(X_test)[:, 1]

print('=== Random forest — test set ===')
print('Accuracy: {:.4f}'.format(accuracy_score(y_test, rf_pred)))
print('Recall:   {:.4f}'.format(recall_score(y_test, rf_pred)))
print('ROC-AUC:  {:.4f}'.format(roc_auc_score(y_test, rf_proba)))

importance = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print()
print('Top three feature importances:')
print(importance.head(3).round(4).to_string())

# Comparison table (test-set metrics; for reference only — selection uses CV).
lr_acc = accuracy_score(y_test, y_pred)
lr_rec = recall_score(y_test, y_pred)
lr_auc = roc_auc_score(y_test, y_proba)
dt3_pred = dt_depth3.predict(X_test)
dt3_proba = dt_depth3.predict_proba(X_test)[:, 1]
dt3_acc = accuracy_score(y_test, dt3_pred)
dt3_rec = recall_score(y_test, dt3_pred)
dt3_auc = roc_auc_score(y_test, dt3_proba)

print()
print('=== Model comparison (test set) ===')
print('| Model                  | Accuracy | Recall | ROC-AUC |')
print('|------------------------|----------|--------|---------|')
print('| Logistic regression    | {:.4f}   | {:.4f} | {:.4f} |'.format(lr_acc, lr_rec, lr_auc))
print('| Depth-3 decision tree  | {:.4f}   | {:.4f} | {:.4f} |'.format(dt3_acc, dt3_rec, dt3_auc))
print('| Random forest          | {:.4f}   | {:.4f} | {:.4f} |'.format(
    accuracy_score(y_test, rf_pred), recall_score(y_test, rf_pred), roc_auc_score(y_test, rf_proba)))
print()
print('Test performance alone is not used to pick the final model (Task 3.5).')


=== Random forest — test set ===
Accuracy: 0.7576
Recall:   0.3469
ROC-AUC:  0.6996

Top three feature importances:
lead_days      0.2205
age            0.1972
distance_km    0.1935

=== Model comparison (test set) ===
| Model                  | Accuracy | Recall | ROC-AUC |
|------------------------|----------|--------|---------|
| Logistic regression    | 0.7879   | 0.3469 | 0.7889 |
| Depth-3 decision tree  | 0.7778   | 0.2245 | 0.6997 |
| Random forest          | 0.7576   | 0.3469 | 0.6996 |

Test performance alone is not used to pick the final model (Task 3.5).


## Task 3.5 — Honest model selection and threshold tuning

Objective: choose the model and the decision threshold **without ever looking at the test set**, then report the test performance exactly once.

Rules followed:
- `GridSearchCV` and all cross-validation run on **training data only**.
- Model selection uses cross-validation scores only.
- Out-of-fold probabilities come from `cross_val_predict` on the training data.
- Threshold is chosen from out-of-fold training predictions.
- The test set is used exactly once, for final reporting.

In [6]:

from sklearn.model_selection import GridSearchCV, cross_val_score, cross_val_predict
from sklearn.pipeline import Pipeline

# ---- Hyperparameter tuning on training data only ----
gs = GridSearchCV(
    RandomForestClassifier(random_state=42),
    {
        'n_estimators': [100, 200],
        'max_depth': [3, 5, 8],
    },
    cv=5,
    scoring='roc_auc',
)
gs.fit(X_train, y_train)
print('=== GridSearchCV (training data only) ===')
print('Best parameters:', gs.best_params_)
print('Best CV ROC-AUC: {:.4f}'.format(gs.best_score_))

# ---- Five-fold cross-validation ROC-AUC comparison (training data only) ----
lr_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000)),
])

cv_models = {
    'Logistic regression': lr_pipe,
    'Depth-3 decision tree': DecisionTreeClassifier(max_depth=3, random_state=42),
    'Random forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Tuned random forest': gs.best_estimator_,
}

print()
print('=== Five-fold CV ROC-AUC (training data only) ===')
cv_results = {}
for name, model in cv_models.items():
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='roc_auc')
    cv_results[name] = float(scores.mean())
    print('{:<26} {:.4f} (+/- {:.4f})'.format(
        name, scores.mean(), scores.std()))

winner_name = max(cv_results, key=cv_results.get)
print()
print('Winner (cross-validation only):', winner_name,
      '| CV ROC-AUC: {:.4f}'.format(cv_results[winner_name]))

# ---- Out-of-fold probabilities and threshold tuning ----
winner = cv_models[winner_name]
oof_proba = cross_val_predict(winner, X_train, y_train, cv=5, method='predict_proba')[:, 1]

print()
print('=== Threshold tuning on out-of-fold training predictions ===')
for thr in [0.50, 0.35]:
    oof_flag = (oof_proba >= thr).astype(int)
    print('Threshold {:.2f}: precision={:.4f} recall={:.4f} F1={:.4f} flagged={}'.format(
        thr,
        precision_score(y_train, oof_flag),
        recall_score(y_train, oof_flag),
        f1_score(y_train, oof_flag),
        int(oof_flag.sum()),
    ))

# Choose the threshold with higher F1.
thresholds = [0.50, 0.35]
threshold_scores = {}
for thr in thresholds:
    oof_flag = (oof_proba >= thr).astype(int)
    threshold_scores[thr] = f1_score(y_train, oof_flag)
selected_threshold = max(threshold_scores, key=threshold_scores.get)
print()
print('Selected threshold: {:.2f} (higher out-of-fold F1: {:.4f})'.format(
    selected_threshold, threshold_scores[selected_threshold]))

print()
print('Recall matters to CareLoop because a missed appointment is a wasted slot and '
      'a lost opportunity to reach the patient. Flagging more at-risk appointments '
      '(higher recall) lets the call centre intervene early, even at the cost of '
      'some false positives.')

# ---- Fit the winner on the training set and evaluate the test set exactly once ----
winner.fit(X_train, y_train)
test_proba = winner.predict_proba(X_test)[:, 1]
test_flag = (test_proba >= selected_threshold).astype(int)

print()
print('=== Final test-set evaluation (threshold {:.2f}) ==='.format(selected_threshold))
print('Precision: {:.4f}'.format(precision_score(y_test, test_flag)))
print('Recall:    {:.4f}'.format(recall_score(y_test, test_flag)))
print('Flagged:   {}'.format(int(test_flag.sum())))
print('ROC-AUC:   {:.4f}'.format(roc_auc_score(y_test, test_proba)))

# ---- Save test risk scores ----
test_risk = pd.DataFrame({
    'appointment_id': id_test.values,
    'no_show': y_test.values,
    'risk_prob': test_proba,
    'flag': test_flag,
})
RISK_CSV = os.path.join(ROOT, 'test_risk_scores.csv')
test_risk.to_csv(RISK_CSV, index=False)
print()
print('Wrote', RISK_CSV, '| rows:', len(test_risk),
      '| flagged:', int(test_risk['flag'].sum()))
print(test_risk.head(10).to_string(index=False))


=== GridSearchCV (training data only) ===
Best parameters: {'max_depth': 3, 'n_estimators': 200}
Best CV ROC-AUC: 0.7674

=== Five-fold CV ROC-AUC (training data only) ===
Logistic regression        0.7737 (+/- 0.0137)
Depth-3 decision tree      0.6784 (+/- 0.0395)


/Users/harish/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/harish/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/harish/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/harish/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/harish/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: overflow encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_

Random forest              0.7452 (+/- 0.0275)


Tuned random forest        0.7674 (+/- 0.0245)

Winner (cross-validation only): Logistic regression | CV ROC-AUC: 0.7737

=== Threshold tuning on out-of-fold training predictions ===
Threshold 0.50: precision=0.5753 recall=0.2485 F1=0.3471 flagged=73
Threshold 0.35: precision=0.5130 recall=0.4675 F1=0.4892 flagged=154

Selected threshold: 0.35 (higher out-of-fold F1: 0.4892)

Recall matters to CareLoop because a missed appointment is a wasted slot and a lost opportunity to reach the patient. Flagging more at-risk appointments (higher recall) lets the call centre intervene early, even at the cost of some false positives.

=== Final test-set evaluation (threshold 0.35) ===
Precision: 0.5714
Recall:    0.5714
Flagged:   49
ROC-AUC:   0.7889

Wrote /Users/harish/careloop-clinics-no-show-guard/test_risk_scores.csv | rows: 198 | flagged: 49
appointment_id  no_show  risk_prob  flag
         A0217        0   0.021565     0
         A0336        0   0.150917     0
         A0508        0   0.51

/Users/harish/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/harish/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/harish/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/harish/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/harish/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: overflow encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_